In [3]:
# Import necessary packages
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
# Read in data: scores for top and bottom 10% by GI* analysis
top_info = pd.read_csv("data/top_10percent_score_trn.csv")
bottom_info = pd.read_csv("data/bottom_10percent_score_trn.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/top_10percent_score_trn.csv'

In [5]:
def standardize_info(df, group_name):
    """Standardize column names and required fields for a top/bottom info table."""
    # Work on a copy so the original dataframe is not modified
    df = df.copy()

    # Rename protein/gene column if it was imported as an unnamed index column
    if "Unnamed: 0" in df.columns and "gene" not in df.columns:
        df = df.rename(columns={"Unnamed: 0": "gene"})

    # Check that required columns are present before continuing
    required_cols = {"gene", "row_max_col", "row_max"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{group_name} is missing required columns: {missing}")

    # Reset index and label rows by group
    df = df.reset_index(drop=True)
    df["group"] = group_name

    return df


def add_ml_logodds(info_df, base_dir):
    """Add ML log2 odds ratio for each protein and its max TRN."""
    # Work on a copy so the input dataframe is preserved
    info_df = info_df.copy().reset_index(drop=True)

    # Store lookup results for each row
    ml_logodds = []
    status = []

    # Look up each protein/TRN pair in the corresponding TRN log-odds file
    for _, row in info_df.iterrows():
        trn = row["row_max_col"]
        gene = row["gene"]
        filepath = base_dir / f"{trn}_log_odds_outputs.csv"

        # Record missing files instead of stopping the full workflow
        if not filepath.exists():
            ml_logodds.append(np.nan)
            status.append("missing_file")
            continue

        # Load log-odds results for this TRN
        trn_df = pd.read_csv(filepath)

        # Pull log2 odds ratio for this gene
        vals = trn_df.loc[trn_df["gene"].eq(gene), "log2_odds_ratio"]

        # Record genes not found in the TRN file
        if vals.empty:
            ml_logodds.append(np.nan)
            status.append("missing_gene")
            continue

        # Use max if higher log-odds indicates stronger positive association
        # Change to vals.min() if strongest negative association is needed instead
        ml_logodds.append(vals.max())
        status.append("found")

    # Add lookup values and status labels to dataframe
    info_df["log_odds_ML"] = ml_logodds
    info_df["ml_lookup_status"] = status

    # Cap placeholder/infinite values for cleaner plotting
    info_df["log_odds_ML"] = info_df["log_odds_ML"].replace({20: 5, -20: -5})

    return info_df


def plot_ml_logodds(info_df, title):
    """Plot sorted ML log2 odds ratios and highlight rows whose max TRN is NRF2."""
    # Drop rows without ML log-odds values and sort from highest to lowest
    plot_df = (
        info_df.dropna(subset=["log_odds_ML"])
        .sort_values(by="log_odds_ML", ascending=False)
        .reset_index(drop=True)
    )

    # Extract plotting values and x positions
    y = plot_df["log_odds_ML"].to_numpy()
    x = np.arange(len(y))

    # Highlight rows where the top TRN is NRF2
    mask = plot_df["row_max_col"].eq("NRF2").to_numpy()
    colors = np.where(mask, "red", "lightgray")

    # Plot sorted ML log-odds values
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.bar(x, y, width=1.0, color=colors)

    # Clean up axis formatting
    ax.set_xticks([])
    ax.set_xlabel("")
    ax.set_ylabel("ML log2 odds ratio")
    ax.set_title(title)
    ax.margins(x=0)

    # Show plot
    plt.tight_layout()
    plt.show()

    # Return sorted dataframe used for plotting
    return plot_df

In [6]:
# Define directory containing TRN-specific ML log-odds output files
base_dir = Path("data/log_odds_outputs")

# Standardize top and bottom info tables separately
top_info = standardize_info(top_info, "top")
bottom_info = standardize_info(bottom_info, "bottom")

# Add ML log2 odds ratios to each group
top_info = add_ml_logodds(top_info, base_dir)
bottom_info = add_ml_logodds(bottom_info, base_dir)

# Plot ML log2 odds ratios for each group, highlighting NRF2-associated rows
top_info_ml = plot_ml_logodds(
    top_info,
    "Top 10% proteins: ML log2 odds ratio, NRF2 highlighted"
)

bottom_info_ml = plot_ml_logodds(
    bottom_info,
    "Bottom 10% proteins: ML log2 odds ratio, NRF2 highlighted"
)

NameError: name 'top_info' is not defined